# Data Cleaning Pipeline

Turns everything in `data/raw/` into analysis-ready files in `data/processed/`.
`02_EDA.ipynb` reads only from `data/processed/` — it never touches `data/raw/`
directly, so any change to cleaning logic belongs here, not there.

**How to run**: top to bottom, "Run All." Every cell is idempotent — re-running
just regenerates the same output files in place. There's no dependency on
`02_EDA.ipynb` having run first, but `02_EDA.ipynb` does depend on this
notebook having run at least once (it reads the files this one writes).

**Inputs** (`data/raw/`):
- 6 Epoch AI CSVs: `data_centers.csv`, `data_center_timelines.csv`,
  `data_center_chip_quantities.csv`, `data_center_chillers.csv`,
  `data_center_cooling_towers.csv`, `gpu_clusters.csv`
- `owid-energy-data.csv` — external, electricity sustainability metrics
- `owid-co2-data.csv` — external, CO2 emissions
- `globalpetrolprices_electricity.csv` — external, electricity prices

**Outputs** (`data/processed/`):
- The 6 `*_clean.csv` counterparts of the Epoch AI files above
- `owid_country_sustainability_clean.csv`
- `owid_co2_clean.csv`
- `electricity_price_clean.csv`

**Structure**: Part A cleans the 6 core datasets with one shared set of rules;
Part B brings in the three external datasets that Part A's files don't cover
(none has a downloadable "clean" source, so each gets compiled/filtered by
hand — see each section for how and why).

## PART A — Cleaning the 6 Core Epoch AI Datasets

One shared rule set (`FILES` dict below) applied to all 6 files: coerce the
listed columns to numeric/datetime, snake_case every column name, write to
`data/processed/`. A couple of files need one extra manual step afterward
(the owner/user confidence-tag split) — that's the last section in this part.

### Setup

Shared imports, the `RAW_DIR`/`OUT_DIR` paths every cell below writes relative to, and two small helpers used throughout Part A: `snake()` (column-name normalizer) and `to_num()` (strips commas/whitespace before coercing to numeric).

In [1]:
import pandas as pd
import re
import os

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

RAW_DIR = "../data/raw"
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

def snake(col):
    col = col.strip()
    col = re.sub(r"[^\w\s]", "", col)
    col = re.sub(r"\s+", "_", col)
    return col.lower()

def to_num(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )

### Quick look at one file before cleaning everything

In [2]:
df = pd.read_csv(f"{RAW_DIR}/data_centers.csv")
df.head()

,Name,Current H100 equivalents,Current power (MW),Current total capital cost (2025 USD billions),Owner,Users,Selected Sources,Calculations sheet,Project,Current chip types,All chip types,Investors,Construction companies,Energy companies,Country,Address
0,Colossus 2,1.111673e+06,946.0,35.836372,SpaceXAI #confident,"Anthropic #confident, Cursor #confident, Space...",- [WSJ profile of the Colossus data centers](h...,https://docs.google.com/spreadsheets/d/1_FQATV...,NaN,"B200,B300","B200,B300",NaN,NaN,NaN,United States,"5420 Tulane Rd, Memphis, TN 38109"
1,Microsoft Fairwater Atlanta,7.687687e+05,636.0,24.092952,Microsoft #confident,"OpenAI #likely, Microsoft #likely",- [2023 Air Permit application](https://drive....,https://docs.google.com/spreadsheets/d/1XeruHu...,Microsoft AI WAN #confident,B200,B200,NaN,NaN,Georgia Power,United States,"1435 Hwy 54 W, Fayetteville, GA 30214"
2,Meta Prometheus,7.630116e+05,631.0,23.903542,Meta #confident,Meta #confident,- [Meta blog post with details on Prometheus](...,https://docs.google.com/spreadsheets/d/1rL6xWR...,Prometheus #confident,B200,B200,NaN,NaN,"Williams,American Electric Power",United States,"1 Community Cir, New Albany, OH 43054"
3,Anthropic-Amazon New Carlisle,6.859121e+05,910.0,34.472620,Amazon #confident,Anthropic #confident,- [Air Construction Permit application for the...,https://docs.google.com/spreadsheets/d/1umU-KL...,Project Rainier #confident,Trainium2,Trainium2,NaN,NaN,NaN,United States,"55001 Larrison Blvd, New Carlisle, IN 46552"
4,OpenAI Stargate Abilene,5.093482e+05,421.0,15.948322,Oracle #confident,OpenAI #confident,"- [Crusoe 2024 Impact Report, with details on ...",https://docs.google.com/spreadsheets/d/197BW-5...,Stargate #confident,"B200,B300","B200,B300","Softbank,OpenAI,Oracle",Mortenson,American Electric Power,United States,"5502 Spinks Rd, Abilene, TX 79601"


### Cleaning rules for all 6 files

`numeric`/`dates` list the columns (by their *original* raw name) that need
coercion in each file — everything else passes through untouched except for
the column-name snake_casing applied to all of them below.

In [3]:
FILES = {
    "data_centers.csv": {
        "numeric": ["Current H100 equivalents", "Current power (MW)",
                    "Current total capital cost (2025 USD billions)"],
        "dates": [],
    },
    "data_center_timelines.csv": {
        "numeric": ["Buildings operational", "IT power (MW)", "Power (MW)",
                    "H100 equivalents", "Performance (8-bit OP/s)",
                    "Total capital cost (2025 USD billions)",
                    "Compute cost (2025 USD billions)",
                    "Construction cost (2025 USD billions)",
                    "Annual operating cost (2025 USD billions)",
                    "Water use (MGD)",
                    "Current H100 equivalents (from Data center)"],
        "dates": ["Date"],
    },
    "data_center_chip_quantities.csv": {
        "numeric": ["Number of Units"],
        "dates": ["Date", "Created", "Last Modified"],
    },
    "data_center_chillers.csv": {
        "numeric": ["Cooling capacity (kW)", "Number of fans",
                    "Length (m)", "Width (m)", "Area (m^2)"],
        "dates": [],
    },
    "data_center_cooling_towers.csv": {
        "numeric": ["Length (m)", "Width (m)", "Height (m)", "Area (m^2)",
                    "Cooling capacity (stated, kW)",
                    "Cooling capacity (central estimate, kW)",
                    "Fan Diameter (m)", "Number of Fans",
                    "Fan estimate diameter (m)"],
        "dates": [],
    },
    "gpu_clusters.csv": {
        "numeric": ["Max OP/s (log)", "H100 equivalents",
                    "Chip quantity (primary)", "Power Capacity (MW)",
                    "Total number of AI chips", "latitude", "longitude"],
        "dates": ["First Operational Date"],
    },
}

### Clean and save all 6 at once

In [4]:
for filename, rules in FILES.items():
    df = pd.read_csv(f"{RAW_DIR}/{filename}")

    for col in rules["numeric"]:
        if col in df.columns:
            df[col] = to_num(df[col])

    for col in rules["dates"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    df.columns = [snake(c) for c in df.columns]

    out_name = filename.replace(".csv", "_clean.csv")
    df.to_csv(f"{OUT_DIR}/{out_name}", index=False)
    print(f"{filename} -> {out_name}  {df.shape}")

data_centers.csv -> data_centers_clean.csv  (74, 16)
data_center_timelines.csv -> data_center_timelines_clean.csv  (419, 14)
data_center_chip_quantities.csv -> data_center_chip_quantities_clean.csv  (196, 14)
data_center_chillers.csv -> data_center_chillers_clean.csv  (143, 12)
data_center_cooling_towers.csv -> data_center_cooling_towers_clean.csv  (527, 17)
gpu_clusters.csv -> gpu_clusters_clean.csv  (482, 57)


### Splitting the owner/user confidence tags

`owner` and `users` columns have a confidence tag baked into the string, e.g.
`"Google #confident"`. Split into a clean name (`owner_name`) and a separate
`owner_confidence` column so downstream analysis can filter or group on
either independently, instead of parsing the combined string every time.

In [5]:

dc_path = f"{OUT_DIR}/data_centers_clean.csv"
dc = pd.read_csv(dc_path)

split_owner = dc["owner"].str.extract(r"^(.*?)\s*#(\w+)$")
dc["owner_name"] = split_owner[0].fillna(dc["owner"])
dc["owner_confidence"] = split_owner[1]

dc.to_csv(dc_path, index=False)
dc[["owner", "owner_name", "owner_confidence"]].head(10)

,owner,owner_name,owner_confidence
0,SpaceXAI #confident,SpaceXAI,confident
1,Microsoft #confident,Microsoft,confident
2,Meta #confident,Meta,confident
3,Amazon #confident,Amazon,confident
4,Oracle #confident,Oracle,confident
5,Microsoft #confident,Microsoft,confident
6,Google #confident,Google,confident
7,Google #confident,Google,confident
8,Google #confident,Google,confident
9,Google #confident,Google,confident


## PART B — External Data Enrichment

Neither dataset below has a clean downloadable source, so each gets manually
filtered/compiled rather than run through the shared `FILES`-dict pattern
from Part A. Both end up with a `country_epoch` column — the country name
spelled the way `gpu_clusters_clean.csv` spells it (formal ISO names for a
handful of countries) — so either file joins onto it directly, the way
`02_EDA.ipynb` does throughout Part 3.

### External data — OWID electricity sustainability metrics

`owid-energy-data.csv` (from https://github.com/owid/energy-data) is a raw
external download, not one of the 6 Epoch AI files above. It has 130 columns
and includes regional/income-group aggregate rows (e.g. "Asia", "World",
"European Union (27)") mixed in with real countries — those aggregates all
have a blank `iso_code`, which is how we filter them out. We only need a
handful of sustainability columns, one row per country (its most recent
reported year), plus a country name aligned to how `data_centers_clean.csv`
and `gpu_clusters_clean.csv` spell country names (they use formal ISO names
for a few countries where OWID uses the common name).

In [6]:
energy_cols = ["country", "iso_code", "year", "renewables_share_elec", "low_carbon_share_elec",
               "fossil_share_elec", "carbon_intensity_elec", "electricity_generation"]
energy = pd.read_csv(f"{RAW_DIR}/owid-energy-data.csv", usecols=energy_cols)

# Drop regional/income-group aggregates (blank iso_code = not a real country).
energy = energy[energy["iso_code"].notna()]

# One row per country: its most recent reported year (years vary by country).
energy = energy.sort_values("year").groupby("country", as_index=False).tail(1)

# Our project's country column uses formal ISO names for a few countries;
# OWID uses the common name. Map OWID -> our spelling so this file joins
# directly onto data_centers_clean.csv / gpu_clusters_clean.csv on country.
epoch_name_map = {
    "South Korea": "Korea (Republic of)",
    "Philippines": "Philippines (the)",
    "United Kingdom": "United Kingdom of Great Britain and Northern Ireland",
    "United States": "United States of America",
}
energy["country_epoch"] = energy["country"].replace(epoch_name_map)

energy.to_csv(f"{OUT_DIR}/owid_country_sustainability_clean.csv", index=False)
print(f"owid-energy-data.csv -> owid_country_sustainability_clean.csv  {energy.shape}")
energy.head()

owid-energy-data.csv -> owid_country_sustainability_clean.csv  (220, 9)


,country,year,iso_code,carbon_intensity_elec,electricity_generation,fossil_share_elec,low_carbon_share_elec,renewables_share_elec,country_epoch
4777,Comoros,2023,COM,642.86,0.14,100.000,0.000,0.000,Comoros
5894,Dominica,2023,DMA,600.00,0.15,86.667,13.333,13.333,Dominica
7579,Faroe Islands,2023,FRO,346.94,0.49,53.061,46.939,46.939,Faroe Islands
7535,Falkland Islands,2023,FLK,1000.00,0.01,100.000,0.000,0.000,Falkland Islands
7855,French Guiana,2023,GUF,244.90,0.98,34.694,65.306,65.306,French Guiana


### External data — electricity prices

`data/raw/globalpetrolprices_electricity.csv` was compiled from the public
country comparison table at
[globalpetrolprices.com/electricity_prices](https://www.globalpetrolprices.com/electricity_prices/)
(residential rate, USD/kWh, 2023-2026 average, accessed 2026-07-18). No free
bulk-download API exists for this data, so this is a manually transcribed
snapshot, not a live feed — treat it as directional, not authoritative.
Country names in the source use abbreviations (`USA`, `UK`, `UAE`) that need
mapping to line up with the rest of this project.

In [7]:
prices = pd.read_csv(f"{RAW_DIR}/globalpetrolprices_electricity.csv")

# Align to the same "country" (OWID/data_centers_clean spelling) and
# "country_epoch" (gpu_clusters_clean spelling) columns used by
# owid_country_sustainability_clean.csv, so all three join the same way.
owid_name_map = {"USA": "United States", "UK": "United Kingdom", "UAE": "United Arab Emirates"}
prices["country"] = prices["country"].replace(owid_name_map)

epoch_name_map = {
    "South Korea": "Korea (Republic of)",
    "Philippines": "Philippines (the)",
    "United Kingdom": "United Kingdom of Great Britain and Northern Ireland",
    "United States": "United States of America",
}
prices["country_epoch"] = prices["country"].replace(epoch_name_map)

prices.to_csv(f"{OUT_DIR}/electricity_price_clean.csv", index=False)
print(f"globalpetrolprices_electricity.csv -> electricity_price_clean.csv  {prices.shape}")
prices.head()

globalpetrolprices_electricity.csv -> electricity_price_clean.csv  (145, 3)


,country,price_usd_per_kwh,country_epoch
0,Bermuda,0.465,Bermuda
1,Ireland,0.450,Ireland
2,Italy,0.414,Italy
3,Cayman Islands,0.410,Cayman Islands
4,Germany,0.406,Germany


### External data — CO2 emissions

`owid-co2-data.csv` (from https://github.com/owid/co2-data, same publisher as
the energy dataset above) was downloaded earlier but never actually used —
none of the sustainability sections needed total emissions, only intensity.
Adding it now to close that gap: total `co2` (million tonnes/year) and
`co2_per_capita`, cleaned the same way as the energy file (drop aggregate
rows via blank `iso_code`, latest year per country, `country_epoch` alias).

In [8]:
co2_cols = ["country", "iso_code", "year", "co2", "co2_per_capita", "population"]
co2 = pd.read_csv(f"{RAW_DIR}/owid-co2-data.csv", usecols=co2_cols)

co2 = co2[co2["iso_code"].notna()]
co2 = co2.sort_values("year").groupby("country", as_index=False).tail(1)
co2["country_epoch"] = co2["country"].replace(epoch_name_map)

co2.to_csv(f"{OUT_DIR}/owid_co2_clean.csv", index=False)
print(f"owid-co2-data.csv -> owid_co2_clean.csv  {co2.shape}")
co2.head()

owid-co2-data.csv -> owid_co2_clean.csv  (218, 7)


,country,year,iso_code,population,co2,co2_per_capita,country_epoch
50410,Zimbabwe,2024,ZWE,16634366.0,13.701,0.824,Zimbabwe
9577,Chad,2024,TCD,20299131.0,2.831,0.139,Chad
44120,Sweden,2024,SWE,10606993.0,38.097,3.592,Sweden
23812,Japan,2024,JPN,123753041.0,961.867,7.772,Japan
32860,Nicaragua,2024,NIC,6916136.0,5.630,0.814,Nicaragua
